# ── Configuração ──────────────────────────────────────────────────────────────
BASE_URL = "https://apmproject-production.up.railway.app"

# Parâmetros verdadeiros (ground truth)
TRUE_BETA  = 2.5      # forma Weibull
TRUE_ETA   = 1000.0   # escala Weibull
N_SAMPLES  = 400      # número de amostras geradas
SEED       = 42
T0         = 600.0    # horímetro atual (para cálculo de R(t) e RUL)
THRESHOLD  = 0.10     # limiar de intervenção (10% de confiabilidade restante)
TOLERANCE  = 0.10     # tolerância aceitável: 10%

In [ ]:
# ── Configuração ──────────────────────────────────────────────────────────────
# Troque para a URL de produção se quiser testar o Railway:
# BASE_URL = "https://apm-app-production.up.railway.app"
BASE_URL = "http://localhost:8002"

# Parâmetros verdadeiros (ground truth)
TRUE_BETA  = 2.5      # forma Weibull
TRUE_ETA   = 1000.0   # escala Weibull
N_SAMPLES  = 400      # número de amostras geradas
SEED       = 42
T0         = 600.0    # horímetro atual (para cálculo de R(t) e RUL)
THRESHOLD  = 0.10     # limiar de intervenção (10% de confiabilidade restante)
TOLERANCE  = 0.10     # tolerância aceitável: 10%

## 1. Valores Teóricos
Calculados diretamente com `scipy` — estes são os **gabaritos**.

In [ ]:
import numpy as np
from scipy.stats import weibull_min
from scipy.special import gamma as gamma_fn
from scipy.optimize import brentq

dist_true = weibull_min(c=TRUE_BETA, scale=TRUE_ETA)

mttf_teorico    = TRUE_ETA * gamma_fn(1 + 1 / TRUE_BETA)
b10_teorico     = dist_true.ppf(0.10)
b50_teorico     = dist_true.ppf(0.50)
r_t0_teorico    = dist_true.sf(T0)

# RUL: tempo até R(T0+t)/R(T0) = THRESHOLD  →  R(T0+t) = THRESHOLD * R(T0)
rul_target = THRESHOLD * r_t0_teorico
rul_teorico = brentq(lambda t: dist_true.sf(T0 + t) - rul_target, 0, 1e6)

print("── Valores Teóricos (scipy) ──────────────────────")
print(f"  MTTF          = {mttf_teorico:.2f} h")
print(f"  B10 Life      = {b10_teorico:.2f} h  (10% falham antes)")
print(f"  B50 Life      = {b50_teorico:.2f} h  (mediana)")
print(f"  R(t={T0:.0f}h)     = {r_t0_teorico:.4f}  ({r_t0_teorico:.1%})")
print(f"  RUL @ {THRESHOLD:.0%}    = {rul_teorico:.2f} h")

## 2. Geração de Dados Sintéticos
Simulamos falhas reais a partir da distribuição verdadeira + ruído gaussiano (3%).

In [ ]:
import requests

# 1. Ajuste de distribuições (β, η, MTTF, B-lives)
resp_fit = requests.post(f"{BASE_URL}/api/v1/analysis/fit", json=records, timeout=60)
resp_fit.raise_for_status()
fit  = resp_fit.json()
best = fit["best"]
print(f"✔  /analysis/fit        → melhor modelo: {best['model_name']}  AICc={best['aicc']:.2f}")

# 2. RUL
rul_payload = {
    "dist_params":   best,
    "current_age":   T0,
    "n_bootstrap":   300,
    "rul_threshold": THRESHOLD,
}
resp_rul = requests.post(f"{BASE_URL}/api/v1/analysis/rul", json=rul_payload, timeout=60)
resp_rul.raise_for_status()
rul = resp_rul.json()
print(f"✔  /analysis/rul        → R(t₀)={rul['r_current']:.4f}  RUL={rul['rul_time']:.1f} h")

# 3. Auditoria (MTTF, B10, B50…)
audit_payload = {
    "records":          records,
    "dist_params":      best,
    "horimetro_atual":  T0,
}
resp_audit = requests.post(f"{BASE_URL}/api/v1/analysis/audit", json=audit_payload, timeout=60)
resp_audit.raise_for_status()
audit = resp_audit.json()
print(f"✔  /analysis/audit      → MTTF={audit['mtbf_h']:.1f} h  B10={audit['b10']:.1f} h")

# 4. ML
ml_payload = {
    "records":         records,
    "horimetro_atual": T0,
    "weibull_params":  best,
}
resp_ml = requests.post(f"{BASE_URL}/api/v1/ml/analyze", json=ml_payload, timeout=60)
resp_ml.raise_for_status()
ml = resp_ml.json()
print(f"✔  /ml/analyze          → Risk Score={ml['risk']['score']}/100")

# 5. Crow-AMSAA
resp_nhpp = requests.post(f"{BASE_URL}/api/v1/analysis/crow-amsaa", json=records, timeout=60)
resp_nhpp.raise_for_status()
nhpp = resp_nhpp.json()
print(f"✔  /analysis/crow-amsaa → β={nhpp.get('beta_hat','?')}  λ={nhpp.get('lambda_hat','?')}")

print("\n✔  Todos os endpoints responderam com sucesso")

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=df['TBF'], nbinsx=40,
    name='TBF simulado',
    marker_color='#00D4FF', opacity=0.7
))
fig.add_vline(x=mttf_teorico, line_dash='dash', line_color='#FF6B6B',
              annotation_text=f"MTTF teórico = {mttf_teorico:.0f} h")
fig.add_vline(x=df['TBF'].mean(), line_dash='dot', line_color='#FFA500',
              annotation_text=f"TBF médio = {df['TBF'].mean():.0f} h")
fig.update_layout(
    title='Distribuição dos TBFs Gerados',
    xaxis_title='TBF (h)', yaxis_title='Frequência',
    template='plotly_dark', height=350
)
fig.show()

## 3. Chamada à API
Enviamos os dados para o endpoint `/api/v1/analysis/full` — o mesmo que o frontend usa.

In [ ]:
import requests

payload = {
    "records":         records,
    "horimetro_atual": T0,
    "tag":             "VALIDATE_001",
    "n_bootstrap":     300,
    "threshold":       THRESHOLD,
}

try:
    resp = requests.post(f"{BASE_URL}/api/v1/analysis/full", json=payload, timeout=60)
    resp.raise_for_status()
    result = resp.json()
    print("✔  API respondeu com sucesso")
except requests.exceptions.ConnectionError:
    print(f"✘  Não foi possível conectar em {BASE_URL}")
    print("   Verifique se o backend está rodando: docker compose up")
    raise
except Exception as e:
    print(f"✘  Erro: {e}")
    raise

## 4. Validação dos Parâmetros Weibull Ajustados
O app usa MLE para estimar β e η — comparamos com os parâmetros verdadeiros.

In [ ]:
fit  = result['fit']
best = fit['best']

print(f"Melhor modelo: {best['model_name']}  (AICc = {best['aicc']:.2f})")
print()

def check(label, estimated, true, tol=TOLERANCE):
    err = abs(estimated - true) / abs(true)
    ok  = err <= tol
    icon = '✔ PASS' if ok else '✘ FAIL'
    print(f"  {icon}  {label:<38}  estimado={estimated:.4f}  teórico={true:.4f}  erro={err:.1%}")
    return ok

passes = []
if best.get('dist_type') == 'weibull':
    passes.append(check('β (forma)',  best['beta'], TRUE_BETA,  tol=0.10))
    passes.append(check('η (escala)', best['eta'],  TRUE_ETA,   tol=0.10))
else:
    print(f"  ⚠  Melhor ajuste: {best['model_name']} (β/η não disponíveis)")
    print("     Tente aumentar N_SAMPLES.")

## 5. Validação das Métricas de Confiabilidade

In [ ]:
audit = result['audit']
rul   = result['rul']

passes.append(check('MTTF (h)',       audit['mtbf_h'],    mttf_teorico))
passes.append(check('B10 Life (h)',   audit['b10'],       b10_teorico))
passes.append(check('B50 Life (h)',   audit['b50'],       b50_teorico))
passes.append(check(f'R(t={T0:.0f}h)', rul['r_current'], r_t0_teorico))
passes.append(check('RUL (h)',        rul['rul_time'],    rul_teorico,  tol=0.15))

## 6. Curva de Sobrevivência — App vs. Teórica
Sobrepomos a curva retornada pela API com a curva teórica de `scipy`.

In [ ]:
# Pega a curva SF retornada pelo app
curves      = {m['model_name']: m for m in fit['models']}
weibull_app = curves.get('Weibull 2P', best)  # fallback para o melhor modelo
t_app       = weibull_app['curve_t']
sf_app      = weibull_app['curve_sf']

# Curva teórica nos mesmos pontos
t_teo  = np.linspace(min(t_app), max(t_app), 300)
sf_teo = dist_true.sf(t_teo)

# Kaplan-Meier
km = fit.get('km_points', {})

fig = go.Figure()

if km:
    fig.add_trace(go.Scatter(
        x=km['t'], y=km['sf'],
        mode='lines+markers', name='Kaplan-Meier (dados)',
        line=dict(color='#FFA500', dash='dot'), marker=dict(size=4)
    ))

fig.add_trace(go.Scatter(
    x=t_app, y=sf_app,
    mode='lines', name='Weibull ajustado (App)',
    line=dict(color='#00D4FF', width=2)
))
fig.add_trace(go.Scatter(
    x=t_teo, y=sf_teo,
    mode='lines', name='Weibull teórico (scipy)',
    line=dict(color='#FF6B6B', width=2, dash='dash')
))
fig.add_vline(x=T0, line_dash='dot', line_color='gray',
              annotation_text=f"t₀ = {T0:.0f} h")

fig.update_layout(
    title='Curva de Sobrevivência — App vs. Teórica',
    xaxis_title='Tempo (h)',
    yaxis_title='R(t) — Probabilidade de Sobrevivência',
    yaxis=dict(range=[0, 1.05]),
    template='plotly_dark', height=420
)
fig.show()

## 7. Sanidade dos Demais Módulos

In [ ]:
def check_bool(label, condition):
    icon = '✔ PASS' if condition else '✘ FAIL'
    print(f"  {icon}  {label}")
    passes.append(condition)
    return condition

ml    = result.get('ml', {})
nhpp  = result.get('nhpp', {})

check_bool('Kaplan-Meier presente',          len(fit.get('km_points', {}).get('t', [])) > 0)
check_bool('≥ 3 modelos LDA retornados',     len(fit.get('models', [])) >= 3)
check_bool('Bootstrap IC P10/P90 presentes', bool(rul.get('ci_p10')) and bool(rul.get('ci_p90')))
check_bool('Crow-AMSAA retornado',           bool(nhpp))
check_bool('ML / Risk Score retornado',      bool(ml))
check_bool('Risk Score entre 0 e 100',       0 <= ml.get('risk', {}).get('score', -1) <= 100)
check_bool('Forecast presente',              bool(ml.get('forecast', {}).get('future_tbfs')))
check_bool('Anomalias detectadas (resultado)', 'anomalies' in ml)
check_bool('Percentis completos no audit',   len(audit.get('percentiles', [])) > 0)

## 8. Resultado Final

In [ ]:
n_pass  = sum(passes)
n_total = len(passes)
pct     = n_pass / n_total * 100

print("=" * 55)
if n_pass == n_total:
    print(f"  ✔  VALIDADO — {n_pass}/{n_total} verificações passaram ({pct:.0f}%)")
else:
    print(f"  ✘  {n_total - n_pass} falha(s) — {n_pass}/{n_total} passaram ({pct:.0f}%)")
print("=" * 55)

# Tabela resumo para colar no LinkedIn / documentação
resumo = pd.DataFrame([
    {'Parâmetro':      'β (forma Weibull)',  'Teórico': TRUE_BETA,        'App': best.get('beta','—')},
    {'Parâmetro':      'η (escala)',         'Teórico': TRUE_ETA,         'App': best.get('eta','—')},
    {'Parâmetro':      'MTTF (h)',           'Teórico': round(mttf_teorico,1), 'App': round(audit.get('mtbf_h',0),1)},
    {'Parâmetro':      'B10 Life (h)',       'Teórico': round(b10_teorico,1),  'App': round(audit.get('b10',0),1)},
    {'Parâmetro':      'B50 Life (h)',       'Teórico': round(b50_teorico,1),  'App': round(audit.get('b50',0),1)},
    {'Parâmetro':      f'R(t={T0:.0f}h)',   'Teórico': round(r_t0_teorico,4), 'App': round(rul.get('r_current',0),4)},
    {'Parâmetro':      'RUL @ 10% (h)',      'Teórico': round(rul_teorico,1),  'App': round(rul.get('rul_time',0),1)},
])
resumo['Erro (%)'] = resumo.apply(
    lambda r: f"{abs(float(r['App']) - float(r['Teórico'])) / abs(float(r['Teórico'])) * 100:.1f}%"
    if r['App'] != '—' else '—', axis=1
)
resumo